# OBCB Model: Verification of PN, PS, PNS Values

Verifies every number quoted in Section 2 of the PCI paper for the **stochastic**
Old Boys' Club Bank (OBCB) example (Example 3 in the paper).

> **Two models in Section 2 — only the stochastic one is implemented here.**
>
> *Deterministic model* (Example 1, Bob at OBCB):
> ```
> loan = ¬check-failed ∧ (gender = male)
> ```
> `check` does **not** appear in the loan formula. An unchecked male would still get
> the loan because `check-failed = check ∧ (credit = bad) = 0` when `check = 0`.
>
> *Stochastic model* (Example 3, OBCB stochastic):
> ```
> loan = loan-if-checked · check
> ```
> `check` **is** required. An unchecked applicant always gets `loan = 0`, regardless
> of gender or credit.
>
> The PN/PS/PNS quantities below are computed for the stochastic model only.

## Model structure

Exogenous variables (independent, Bernoulli(0.5) each):
- `gender`: 0 = female, 1 = male
- `credit`: 0 = bad, 1 = good

Endogenous variables (structural equations below):
- `check`         ~ Bern(0.9) if male, Bern(0.2) if female
- `check_failed`  = `check` × (1 − `credit`)
- `loan_if_checked` ~ Bern(`loan_prob[gender, check_failed]`)
- `loan`          = `loan_if_checked` × `check`

The `loan_prob` matrix:

| | check_failed=0 | check_failed=1 |
|---|---|---|
| female | 0.9 | 0.0 |
| male   | 1.0 | 0.05 |

## Computation approach

All computations use the **partial abduction** formula consistent with how
Pearl's PN/PS/PNS are typically computed for Markovian (acyclic, no hidden
confounders) SCMs: context variables are conditioned on, and the marginal
interventional probability `P(Y | do(C=c), context)` is used directly.
This matches the numbers stated in the paper.

Concretely:
- **PN(C=c\*, Y=0)** = P(loan=1 | do(C=c'), context) averaged over the
  posterior P(context | C=c\*, loan=0)
- **PS(C=c\*, Y=0)** = P(loan=0 | do(C=c\*), context) averaged over the
  posterior P(context | C=c', loan=1)
- **PNS(C=c\*, Y=0)** = P(loan=0 | C=c\*, context) × P(loan=1 | C=c', context)
  averaged over the prior P(context)

Individual-level quantities fix the context (Alice = female/bad, Bob = male/bad)
so the posterior average collapses to a single term.

In [1]:
import numpy as np
import pandas as pd

# ── Model parameters ─────────────────────────────────────────────────────────
# P(check=1 | gender)
p_check = {0: 0.2, 1: 0.9}   # 0=female, 1=male

# P(loan_if_checked=1 | gender, check_failed)
# check_failed = check * (1-credit); equals 1 only when check=1 AND credit=bad.
loan_prob = {
    (0, 0): 0.9,    # female, check_failed=0  (checked+good OR unchecked, but see note)
    (0, 1): 0.0,    # female, check_failed=1  (checked+bad → always denied)
    (1, 0): 1.0,    # male,   check_failed=0
    (1, 1): 0.05,   # male,   check_failed=1  (checked+bad → 5% approved)
}

def p_loan(gender, credit):
    """P(loan=1 | gender, credit) for the STOCHASTIC model.

    Structural equation: loan = loan_if_checked * check
    So loan=1 requires check=1.  When check=0, loan=0 regardless of credit.

    Decomposition:
      P(loan=1) = P(check=1) * P(loan_if_checked=1 | check=1, credit)

    Given check=1: check_failed = 1 * (1-credit) = 1-credit, so we index
    loan_prob by (gender, 1-credit).  The check=0 branch contributes 0.
    """
    check_failed_given_checked = 1 - credit
    return p_check[gender] * loan_prob[(gender, check_failed_given_checked)]

# Sanity check: print all four marginal loan probabilities
print("P(loan=1 | gender, credit):")
for g, gl in [(0,'female'), (1,'male')]:
    for c, cl in [(0,'bad'), (1,'good')]:
        print(f"  {gl:6s}, {cl:4s}: {p_loan(g,c):.4f}")

P(loan=1 | gender, credit):
  female, bad : 0.0000
  female, good: 0.1800
  male  , bad : 0.0450
  male  , good: 0.9000


## Probability of Necessity (PN)

$$\mathrm{PN}(C=c^*, Y=0) = P(\mathrm{loan}_{C=c'} = 1 \mid C=c^*, Y=0)$$

"Had the cause been absent, would the outcome still have occurred?"

**Population-level:** marginalise over credit using the posterior
$P(\text{credit} \mid C=c^*, \text{loan}=0)$ — i.e. the distribution of credit
among rejected applicants with the observed gender.

**Individual-level:** credit is fixed by the individual's context, so
$\mathrm{PN}(C=c^* \mid \text{context}) = P(\text{loan}=1 \mid \text{do}(C=c'),\, \text{context})$
with no further averaging.

In [2]:
# ── Helper: posterior on credit within a (gender, loan=0) stratum ─────────────

def p_joint_loan0(gender, credit):
    """P(gender, credit, loan=0) — assuming P(gender)=P(credit)=0.5."""
    return 0.5 * 0.5 * (1 - p_loan(gender, credit))

def p_credit_given_gender_loan0(credit, gender):
    """P(credit | gender, loan=0) — posterior on credit for rejected applicants."""
    numerator = p_joint_loan0(gender, credit)
    denominator = sum(p_joint_loan0(gender, c) for c in [0, 1])
    return numerator / denominator

print("P(credit | gender, loan=0) — posterior credit distribution among rejected applicants:")
for g, gl in [(0,'female'), (1,'male')]:
    for c, cl in [(0,'bad'), (1,'good')]:
        print(f"  {gl:6s}, {cl:4s}: {p_credit_given_gender_loan0(c,g):.4f}")

P(credit | gender, loan=0) — posterior credit distribution among rejected applicants:
  female, bad : 0.5495
  female, good: 0.4505
  male  , bad : 0.9052
  male  , good: 0.0948


In [3]:
# ── Population PN: gender ─────────────────────────────────────────────────────
# PN(gender=g, loan=F) = E_{credit ~ posterior(g, loan=0)}[ P(loan=1 | do(gender=g'), credit) ]

def pn_gender(observed_gender, counterfactual_gender):
    return sum(
        p_credit_given_gender_loan0(c, observed_gender) * p_loan(counterfactual_gender, c)
        for c in [0, 1]
    )

pn_male   = pn_gender(observed_gender=1, counterfactual_gender=0)
pn_female = pn_gender(observed_gender=0, counterfactual_gender=1)

print(f"PN(gender=male,   loan=F) = {pn_male:.4f}   paper: ≈ 0.02")
print(f"PN(gender=female, loan=F) = {pn_female:.4f}   paper: ≈ 0.43")

# ── Population PN: credit ─────────────────────────────────────────────────────
# Symmetric construction: condition on (credit=observed, loan=0) and marginalise over gender.

def p_gender_given_credit_loan0(gender, credit):
    """P(gender | credit, loan=0)."""
    numerator = p_joint_loan0(gender, credit)
    denominator = sum(p_joint_loan0(g, credit) for g in [0, 1])
    return numerator / denominator

def pn_credit(observed_credit, counterfactual_credit):
    return sum(
        p_gender_given_credit_loan0(g, observed_credit) * p_loan(g, counterfactual_credit)
        for g in [0, 1]
    )

pn_credit_bad = pn_credit(observed_credit=0, counterfactual_credit=1)
print(f"PN(credit=bad,    loan=F) = {pn_credit_bad:.4f}   paper: ≈ 0.45")

PN(gender=male,   loan=F) = 0.0171   paper: ≈ 0.02
PN(gender=female, loan=F) = 0.4302   paper: ≈ 0.43
PN(credit=bad,    loan=F) = 0.5317   paper: ≈ 0.45


In [4]:
# ── Individual PN: Alice (female, bad credit) ─────────────────────────────────
#
# Context is fixed: (gender=female, credit=bad). No averaging needed.
#
# PN(gender=female | Alice) = P(loan=1 | do(gender=male), credit=bad)
pn_alice_gender = p_loan(gender=1, credit=0)   # male, bad credit

# PN(credit=bad | Alice) = P(loan=1 | do(credit=good), gender=female)
pn_alice_credit = p_loan(gender=0, credit=1)   # female, good credit

print(f"PN(gender=female | Alice) = {pn_alice_gender:.4f}   paper: 0.045")
print(f"PN(credit=bad    | Alice) = {pn_alice_credit:.4f}   paper: 0.18")

# ── Individual PN: Bob (male, bad credit) ─────────────────────────────────────
#
# PN(gender=male | Bob) = P(loan=1 | do(gender=female), credit=bad)
# loan_prob[female, check_failed=1] = 0.0, so this is exactly 0.
# The value 0.002 in the original draft table was a Monte Carlo artifact.
pn_bob_gender = p_loan(gender=0, credit=0)     # female, bad credit

# PN(credit=bad | Bob) = P(loan=1 | do(credit=good), gender=male)
pn_bob_credit = p_loan(gender=1, credit=1)     # male, good credit

print(f"\nPN(gender=male   | Bob)   = {pn_bob_gender:.4f}   paper: 0.002  ← DISCREPANCY (exact = 0)")
print(f"PN(credit=bad    | Bob)   = {pn_bob_credit:.4f}   paper: 0.90")

PN(gender=female | Alice) = 0.0450   paper: 0.045
PN(credit=bad    | Alice) = 0.1800   paper: 0.18

PN(gender=male   | Bob)   = 0.0000   paper: 0.002  ← DISCREPANCY (exact = 0)
PN(credit=bad    | Bob)   = 0.9000   paper: 0.90


## Probability of Sufficiency (PS)

$$\mathrm{PS}(C=c^*, Y=0) = P(\mathrm{loan}_{C=c^*}=0 \mid C=c',\, Y=1)$$

"Had the cause been present when it was absent, would the outcome have occurred?"

**Population-level:** marginalise over credit using the posterior
$P(\text{credit} \mid C=c', \text{loan}=1)$ — the credit distribution among
*approved* applicants with the counterfactual gender.

**Individual-level:** context is fixed, so
$\mathrm{PS}(C=c^* \mid \text{context}) = P(\text{loan}=0 \mid \text{do}(C=c^*),\, \text{context})$,
provided the conditioning event $\{C=c', Y=1\}$ has positive probability.

> **Note on abduction:** This uses the *partial* abduction convention — context
> variables (gender, credit) are conditioned on but noise for other endogenous
> variables (check) is marginalised out, not fixed by the observation of
> `loan=1`. This is what gives the population values quoted in the paper
> (e.g. PS(credit=bad) ≈ 0.96). Under *full* abduction, conditioning on
> `loan=1` would additionally fix `check=1`, giving slightly different
> individual-level values.

In [5]:
# ── Helper: posterior on credit within a (gender, loan=1) stratum ─────────────

def p_credit_given_gender_loan1(credit, gender):
    """P(credit | gender, loan=1) — posterior credit among approved applicants."""
    numerator   = 0.5 * p_loan(gender, credit)
    denominator = sum(0.5 * p_loan(gender, c) for c in [0, 1])
    return numerator / denominator

# ── Population PS: gender ─────────────────────────────────────────────────────
# PS(gender=g, loan=F) = E_{credit ~ posterior(g', loan=1)}[ P(loan=0 | do(gender=g), credit) ]

def ps_gender(factual_gender, counterfactual_gender):
    return sum(
        p_credit_given_gender_loan1(c, counterfactual_gender) * (1 - p_loan(factual_gender, c))
        for c in [0, 1]
    )

ps_male   = ps_gender(factual_gender=1, counterfactual_gender=0)
ps_female = ps_gender(factual_gender=0, counterfactual_gender=1)

print(f"PS(gender=male,   loan=F) = {ps_male:.4f}   paper: ≈ 0.10")
print(f"PS(gender=female, loan=F) = {ps_female:.4f}   paper: ≈ 0.83")

# ── Population PS: credit ─────────────────────────────────────────────────────

def p_gender_given_credit_loan1(gender, credit):
    """P(gender | credit, loan=1)."""
    numerator   = 0.5 * p_loan(gender, credit)
    denominator = sum(0.5 * p_loan(g, credit) for g in [0, 1])
    return numerator / denominator

def ps_credit(factual_credit, counterfactual_credit):
    return sum(
        p_gender_given_credit_loan1(g, counterfactual_credit) * (1 - p_loan(g, factual_credit))
        for g in [0, 1]
    )

ps_credit_bad = ps_credit(factual_credit=0, counterfactual_credit=1)
print(f"PS(credit=bad,    loan=F) = {ps_credit_bad:.4f}   paper: ≈ 0.96")

PS(gender=male,   loan=F) = 0.1000   paper: ≈ 0.10
PS(gender=female, loan=F) = 0.8286   paper: ≈ 0.83
PS(credit=bad,    loan=F) = 0.9625   paper: ≈ 0.96


In [6]:
# ── Individual PS: Alice (female, bad credit) ─────────────────────────────────
#
# PS(gender=female | Alice) = P(loan=0 | do(gender=female), credit=bad)
#   Conditioning event: (gender=male, credit=bad, loan=1).
#   P(loan=1 | male, bad) = 0.045 > 0, so this is well-defined.
#   After fixing context (female, bad), P(loan=1) = 0.0 → PS = 1.
ps_alice_gender = 1 - p_loan(gender=0, credit=0)   # = 1 - 0 = 1.0

# PS(credit=bad | Alice) = P(loan=0 | do(credit=bad), gender=female)
#   Conditioning event: (credit=good, gender=female, loan=1).
#   P(loan=1 | female, good) = 0.18 > 0, so this is well-defined.
#   After fixing context (female, bad), P(loan=1) = 0.0 → PS = 1.
ps_alice_credit = 1 - p_loan(gender=0, credit=0)   # = 1 - 0 = 1.0

print(f"PS(gender=female | Alice) = {ps_alice_gender:.4f}   paper: 1.00")
print(f"PS(credit=bad    | Alice) = {ps_alice_credit:.4f}   paper: 1.00")

# ── Individual PS: Bob (male, bad credit) ─────────────────────────────────────
#
# PS(gender=male | Bob):
#   Conditioning event requires (gender=female, credit=bad, loan=1).
#   P(loan=1 | female, bad) = 0 → conditioning event is impossible → UNDEFINED.
ps_bob_gender_conditioning_prob = p_loan(gender=0, credit=0)
print(f"\nPS(gender=male | Bob): conditioning event P = {ps_bob_gender_conditioning_prob} → UNDEFINED")

# PS(credit=bad | Bob) = P(loan=0 | do(credit=bad), gender=male)
#   Conditioning event: (credit=good, gender=male, loan=1).
#   P(loan=1 | male, good) = 0.9 > 0, so this is well-defined.
#   P(loan=1 | male, bad) = 0.9 × 0.05 = 0.045, so P(loan=0 | male, bad) = 0.955.
#   The table value of 1.00 is incorrect — a male with bad credit still has a 4.5%
#   chance of approval (loan_prob[male, check_failed=1] = 0.05).
ps_bob_credit = 1 - p_loan(gender=1, credit=0)   # = 1 - 0.045 = 0.955
print(f"PS(credit=bad    | Bob)   = {ps_bob_credit:.4f}   paper: 1.00  ← DISCREPANCY")

PS(gender=female | Alice) = 1.0000   paper: 1.00
PS(credit=bad    | Alice) = 1.0000   paper: 1.00

PS(gender=male | Bob): conditioning event P = 0.0 → UNDEFINED
PS(credit=bad    | Bob)   = 0.9550   paper: 1.00  ← DISCREPANCY


## Probability of Necessity and Sufficiency (PNS)

$$\mathrm{PNS}(C=c^*, Y=0) = P(Y_{c^*}=0,\; Y_{c'}=1)$$

"Would the outcome occur in the factual world but not in the counterfactual?"

Under the independence assumption (noise variables for different units are
independent), this factorises as:

$$\mathrm{PNS} = \sum_{\text{context}} P(\text{context}) \cdot P(Y=0 \mid C=c^*,\, \text{context}) \cdot P(Y=1 \mid C=c',\, \text{context})$$

**Individual-level (conditional PNS):** fix context to the individual's values.
Then $\mathrm{PNS}_c(C=c^* \mid \text{context}) = P(Y=0 \mid C=c^*,\, \text{context}) \times P(Y=1 \mid C=c',\, \text{context})$.

In [7]:
# ── Population PNS: gender ────────────────────────────────────────────────────
# PNS(gender=g, loan=F) = E_{credit}[ P(loan=0 | g, credit) × P(loan=1 | g', credit) ]

def pns_gender(factual_gender, counterfactual_gender):
    return sum(
        0.5 * (1 - p_loan(factual_gender, c)) * p_loan(counterfactual_gender, c)
        for c in [0, 1]
    )

pns_male   = pns_gender(factual_gender=1, counterfactual_gender=0)
pns_female = pns_gender(factual_gender=0, counterfactual_gender=1)

print(f"PNS(gender=male,   loan=F) = {pns_male:.4f}   paper: ≈ 0.01")
print(f"PNS(gender=female, loan=F) = {pns_female:.4f}   paper: ≈ 0.39")

# ── Population PNS: credit ────────────────────────────────────────────────────
# PNS(credit=c, loan=F) = E_{gender}[ P(loan=0 | gender, c) × P(loan=1 | gender, c') ]

def pns_credit(factual_credit, counterfactual_credit):
    return sum(
        0.5 * (1 - p_loan(g, factual_credit)) * p_loan(g, counterfactual_credit)
        for g in [0, 1]
    )

pns_credit_bad = pns_credit(factual_credit=0, counterfactual_credit=1)
print(f"PNS(credit=bad,    loan=F) = {pns_credit_bad:.4f}   paper: ≈ 0.52")

PNS(gender=male,   loan=F) = 0.0090   paper: ≈ 0.01
PNS(gender=female, loan=F) = 0.3915   paper: ≈ 0.39
PNS(credit=bad,    loan=F) = 0.5197   paper: ≈ 0.52


In [8]:
# ── Individual PNS: Alice (female, bad credit) ────────────────────────────────
#
# PNS_c(gender=female | credit=bad) = P(loan=0|female,bad) × P(loan=1|male,bad)
pns_alice_gender = (1 - p_loan(0, 0)) * p_loan(1, 0)

# PNS_c(credit=bad | gender=female) = P(loan=0|female,bad) × P(loan=1|female,good)
pns_alice_credit = (1 - p_loan(0, 0)) * p_loan(0, 1)

print(f"PNS_c(gender=female | credit=bad)  [Alice] = {pns_alice_gender:.4f}   paper: 0.045")
print(f"PNS_c(credit=bad    | gender=female) [Alice] = {pns_alice_credit:.4f}   paper: 0.18")

# ── Individual PNS: Bob (male, bad credit) ───────────────────────────────────
#
# PNS_c(gender=male | credit=bad) = P(loan=0|male,bad) × P(loan=1|female,bad)
#   P(loan=1 | female, bad) = 0 exactly, so PNS = 0.
pns_bob_gender = (1 - p_loan(1, 0)) * p_loan(0, 0)

# PNS_c(credit=bad | gender=male) = P(loan=0|male,bad) × P(loan=1|male,good)
#   = 0.955 × 0.9 = 0.8595 → rounds to 0.86, not 0.85 as in the paper.
pns_bob_credit = (1 - p_loan(1, 0)) * p_loan(1, 1)

print(f"\nPNS_c(gender=male   | credit=bad)  [Bob]   = {pns_bob_gender:.4f}    paper: 0")
print(f"PNS_c(credit=bad    | gender=male) [Bob]   = {pns_bob_credit:.4f}   paper: 0.85  ← DISCREPANCY (rounds to 0.86)")

PNS_c(gender=female | credit=bad)  [Alice] = 0.0450   paper: 0.045
PNS_c(credit=bad    | gender=female) [Alice] = 0.1800   paper: 0.18

PNS_c(gender=male   | credit=bad)  [Bob]   = 0.0000    paper: 0
PNS_c(credit=bad    | gender=male) [Bob]   = 0.8595   paper: 0.85  ← DISCREPANCY (rounds to 0.86)


## Summary

In [9]:
rows = [
    # Population-level
    ("Population", "PN(gender=male)",    pn_male,        "≈ 0.02", "✓"),
    ("Population", "PN(gender=female)",  pn_female,      "≈ 0.43", "✓"),
    ("Population", "PN(credit=bad)",     pn_credit_bad,  "≈ 0.45", "⚠ exact=0.532"),
    ("Population", "PS(gender=male)",    ps_male,        "≈ 0.10", "✓"),
    ("Population", "PS(gender=female)",  ps_female,      "≈ 0.83", "✓"),
    ("Population", "PS(credit=bad)",     ps_credit_bad,  "≈ 0.96", "✓"),
    ("Population", "PNS(gender=male)",   pns_male,       "≈ 0.01", "✓"),
    ("Population", "PNS(gender=female)", pns_female,     "≈ 0.39", "✓"),
    ("Population", "PNS(credit=bad)",    pns_credit_bad, "≈ 0.52", "✓"),
    # Alice
    ("Alice", "PN(gender=female)",        pn_alice_gender,  "0.045", "✓"),
    ("Alice", "PN(credit=bad)",           pn_alice_credit,  "0.18",  "✓"),
    ("Alice", "PS(gender=female)",        ps_alice_gender,  "1.00",  "✓"),
    ("Alice", "PS(credit=bad)",           ps_alice_credit,  "1.00",  "✓"),
    ("Alice", "PNS_c(gender|bad)",        pns_alice_gender, "0.045", "✓"),
    ("Alice", "PNS_c(credit|female)",     pns_alice_credit, "0.18",  "✓"),
    # Bob
    ("Bob", "PN(gender=male)",            pn_bob_gender,    "0.002", "⚠ exact=0"),
    ("Bob", "PN(credit=bad)",             pn_bob_credit,    "0.90",  "✓"),
    ("Bob", "PS(gender=male)",            float('nan'),     "---",   "✓ (undefined)"),
    ("Bob", "PS(credit=bad)",             ps_bob_credit,    "1.00",  "⚠ exact=0.955"),
    ("Bob", "PNS_c(gender|bad)",          pns_bob_gender,   "0",     "✓"),
    ("Bob", "PNS_c(credit|male)",         pns_bob_credit,   "0.85",  "⚠ exact=0.860"),
]

df = pd.DataFrame(rows, columns=["Context", "Quantity", "Computed", "Paper", "Status"])
df["Computed"] = df["Computed"].map(lambda x: f"{x:.4f}" if not pd.isna(x) else "undef")

pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 120)
print(df.to_string(index=False))

   Context             Quantity Computed  Paper        Status
Population      PN(gender=male)   0.0171 ≈ 0.02             ✓
Population    PN(gender=female)   0.4302 ≈ 0.43             ✓
Population       PN(credit=bad)   0.5317 ≈ 0.45 ⚠ exact=0.532
Population      PS(gender=male)   0.1000 ≈ 0.10             ✓
Population    PS(gender=female)   0.8286 ≈ 0.83             ✓
Population       PS(credit=bad)   0.9625 ≈ 0.96             ✓
Population     PNS(gender=male)   0.0090 ≈ 0.01             ✓
Population   PNS(gender=female)   0.3915 ≈ 0.39             ✓
Population      PNS(credit=bad)   0.5197 ≈ 0.52             ✓
     Alice    PN(gender=female)   0.0450  0.045             ✓
     Alice       PN(credit=bad)   0.1800   0.18             ✓
     Alice    PS(gender=female)   1.0000   1.00             ✓
     Alice       PS(credit=bad)   1.0000   1.00             ✓
     Alice    PNS_c(gender|bad)   0.0450  0.045             ✓
     Alice PNS_c(credit|female)   0.1800   0.18             ✓
       B

## Discrepancies to fix in the paper

Four values need correction:

| Cell | Paper | Correct | Reason |
|------|-------|---------|--------|
| Pop PN(credit=bad) | ≈ 0.45 | 0.532 | Posterior over gender shifts toward female (P≈0.51); weighted sum gives 0.532, not 0.45 |
| Bob PN(gender) | 0.002 | 0 | `loan_prob[female, check_failed=1] = 0` exactly; 0.002 was a MC artifact |
| Bob PS(credit) | 1.00 | 0.955 | Male with bad credit still approved 4.5% of time; `1 − 0.9×0.05 = 0.955` |
| Bob PNS(credit) | 0.85 | 0.860 | `0.955 × 0.9 = 0.8595`, rounds to 0.86 not 0.85 |

All other 17 values verify correctly against the paper.

## PCI: Probabilistic Causal Impact for Alice and Bob

Implements PCI for the OBCB stochastic model with:
- **Suspects** S = {gender, credit}
- **Witnesses** W = {check_failed}
- **Variable selection** Γ_s: uniform over non-empty subsets of S (weight 1/3 each)
- **Witness selection** Γ_w: uniform over {∅, {check_failed}} (weight 1/2 each)
- **Alternative values** Δ: point mass on the only other binary value
- **ci function**: PNS binary — E[ci] = P(Y^n=1, Y^s=0) with y★=loan=0

### U_check enumeration

The exogenous noise U_check ~ Uniform(0,1) determines the check value for each gender.
Three canonical regions capture all distinct counterfactual check combinations:

| Region | Probability | check(female) | check(male) |
|--------|-------------|---------------|-------------|
| u1: U_check ≤ 0.2   | 0.2 | 1 | 1 |
| u2: 0.2 < U_check ≤ 0.9 | 0.7 | 0 | 1 |
| u3: U_check > 0.9   | 0.1 | 0 | 0 |

Region **u2** is the critical one: Alice is not checked as female but would be as male.
The witness on check_failed is what blocks credit's causal path in u1 (where Alice was checked and failed).

In [10]:
# ── U_check regions ──────────────────────────────────────────────────────────
# Each region is (probability, {gender: check_value}).
# These three regions cover all distinct (check_female, check_male) combinations
# given the two thresholds p_check[0]=0.2 and p_check[1]=0.9.

u_regions = {
    'u1': (0.2, {0: 1, 1: 1}),   # U_check ≤ 0.2:       both genders checked
    'u2': (0.7, {0: 0, 1: 1}),   # 0.2 < U_check ≤ 0.9: only male checked
    'u3': (0.1, {0: 0, 1: 0}),   # U_check > 0.9:        neither checked
}

def p_loan_do(gender_int, credit_int, check_val, witness_cf=None):
    """
    P(loan=1) under do(gender=gender_int, credit=credit_int),
    with optional witness do(check_failed=witness_cf).
    check_val is the realised check for gender_int in the current u-region.
    If witness_cf is None, check_failed is computed structurally.
    """
    if check_val == 0:
        return 0.0
    cf = witness_cf if witness_cf is not None else check_val * (1 - credit_int)
    return loan_prob[(gender_int, cf)]


def compute_pci(factual_gender, factual_credit, use_witnesses=True):
    """
    Compute E[ci_gender] and E[ci_credit] for an individual with the given
    factual context, using the PNS binary ci function (y★ = loan = 0).

    Γ_s: uniform over {{gender}, {credit}, {gender,credit}}  (1/3 each)
    Γ_w: uniform over {∅, {check_failed}}                    (1/2 each, or 1 if no witnesses)
    Δ:   point mass on the only other binary value
    """
    alt_gender = 1 - factual_gender
    alt_credit = 1 - factual_credit

    # For each suspect subset, store (alt_g, alt_c) — the alternative values
    # used in the necessity intervention.
    subsets = {
        'gender': (alt_gender, factual_credit),   # only gender changes
        'credit': (factual_gender, alt_credit),   # only credit changes
        'both':   (alt_gender, alt_credit),        # both change
    }
    p_s = 1 / 3

    witness_options = [None, 'check_failed'] if use_witnesses else [None]
    p_w = 1 / len(witness_options)

    results = {}
    for target in ['gender', 'credit']:
        # 2^S_k: subsets of S that contain the target variable
        relevant = ['gender', 'both'] if target == 'gender' else ['credit', 'both']

        total = 0.0
        for region_name, (region_prob, check_by_gender) in u_regions.items():
            factual_check = check_by_gender[factual_gender]
            factual_cf    = factual_check * (1 - factual_credit)

            P_s_meas = 0.0  # sufficiency measure on {loan=0}
            P_n_meas = 0.0  # necessity measure on {loan=1}

            for subset in relevant:
                alt_g, alt_c = subsets[subset]
                for witness in witness_options:
                    w_val = factual_cf if witness == 'check_failed' else None

                    # ── Sufficiency: do(C = factual values, T = factual) ──────
                    suf_check = check_by_gender[factual_gender]
                    p_suf = 1.0 - p_loan_do(factual_gender, factual_credit, suf_check, w_val)
                    P_s_meas += p_s * p_w * p_suf

                    # ── Necessity: do(C = alternative values, T = factual) ────
                    # If C contains gender, the counterfactual gender is alt_g,
                    # which changes the check value via the structural equation.
                    nec_gender_for_check = alt_g if subset in ('gender', 'both') else factual_gender
                    nec_check = check_by_gender[nec_gender_for_check]
                    p_nec = p_loan_do(alt_g, alt_c, nec_check, w_val)
                    P_n_meas += p_s * p_w * p_nec

            total += region_prob * P_s_meas * P_n_meas

        results[target] = total

    return results['gender'], results['credit']

In [11]:
# ── Compute PCI for all four cases ───────────────────────────────────────────

alice_g_w,  alice_c_w  = compute_pci(factual_gender=0, factual_credit=0, use_witnesses=True)
alice_g_nw, alice_c_nw = compute_pci(factual_gender=0, factual_credit=0, use_witnesses=False)
bob_g_w,    bob_c_w    = compute_pci(factual_gender=1, factual_credit=0, use_witnesses=True)
bob_g_nw,   bob_c_nw   = compute_pci(factual_gender=1, factual_credit=0, use_witnesses=False)

# PNS individual values (from earlier cells)
pns_alice_gender = pns_alice_gender   # 0.045
pns_alice_credit = pns_alice_credit   # 0.18
pns_bob_gender   = pns_bob_gender     # 0.0
pns_bob_credit   = pns_bob_credit     # 0.8595

rows = [
    ("Alice", "gender", f"{pns_alice_gender:.4f}", f"{alice_g_nw:.4f}", f"{alice_g_w:.4f}"),
    ("Alice", "credit", f"{pns_alice_credit:.4f}", f"{alice_c_nw:.4f}", f"{alice_c_w:.4f}"),
    ("Bob",   "gender", f"{pns_bob_gender:.4f}",   f"{bob_g_nw:.4f}",   f"{bob_g_w:.4f}"),
    ("Bob",   "credit", f"{pns_bob_credit:.4f}",   f"{bob_c_nw:.4f}",   f"{bob_c_w:.4f}"),
]

df = pd.DataFrame(rows, columns=["Person", "Feature", "PNS", "PCI (no witnesses)", "PCI (with witnesses)"])
print(df.to_string(index=False))
print()

# Ranking judgements
print("Rankings (gender vs credit):")
print(f"  PNS   — Alice: gender {'>' if pns_alice_gender > pns_alice_credit else '<'} credit  (correct: gender > credit)  {'✓' if pns_alice_gender > pns_alice_credit else '✗'}")
print(f"  PNS   — Bob:   gender {'>' if pns_bob_gender   > pns_bob_credit   else '<'} credit  (correct: credit > gender)  {'✓' if pns_bob_gender < pns_bob_credit else '✗'}")
print(f"  PCI w/o witnesses — Alice: gender {'>' if alice_g_nw > alice_c_nw else '<'} credit  {'✓' if alice_g_nw > alice_c_nw else '✗'}")
print(f"  PCI w/o witnesses — Bob:   gender {'>' if bob_g_nw   > bob_c_nw   else '<'} credit  {'✓' if bob_g_nw < bob_c_nw else '✗'}")
print(f"  PCI with witnesses — Alice: gender {'>' if alice_g_w > alice_c_w else '<'} credit  {'✓' if alice_g_w > alice_c_w else '✗'}")
print(f"  PCI with witnesses — Bob:   gender {'>' if bob_g_w   > bob_c_w   else '<'} credit  {'✓' if bob_g_w < bob_c_w else '✗'}")

Person Feature    PNS PCI (no witnesses) PCI (with witnesses)
 Alice  gender 0.0450             0.2100               0.2628
 Alice  credit 0.1800             0.2400               0.1989
   Bob  gender 0.0000             0.0380               0.0190
   Bob  credit 0.8595             0.2280               0.1187

Rankings (gender vs credit):
  PNS   — Alice: gender < credit  (correct: gender > credit)  ✗
  PNS   — Bob:   gender < credit  (correct: credit > gender)  ✓
  PCI w/o witnesses — Alice: gender < credit  ✗
  PCI w/o witnesses — Bob:   gender < credit  ✓
  PCI with witnesses — Alice: gender > credit  ✓
  PCI with witnesses — Bob:   gender < credit  ✓


## Verification: PCI recovers PNS when $|\mathbf{S}|=1$ and $\mathbf{W}=\emptyset$

The paper's property (i) claims that PCI recovers Pearl's PNS exactly when:
- $\mathbf{S} = \{X_k\}$ — a single suspect,
- $\mathbf{W} = \emptyset$ — no witnesses,
- $\dom(X_k) = \{0, 1\}$ — binary,

with the PNS binary $ci(y^s,y^n,y^\star) = \mathbb{1}\{y^n \neq y^\star\}\,\mathbb{1}\{y^s = y^\star\}$.

Under those conditions $\Gamma$-averaging collapses (only one $(C, T)$ configuration), and PCI reduces to

$$\mathrm{PCI}(X_k) \;=\; \int P\!\left(Y = y^\star \,\big|\, u, \mathrm{do}(X_k = x_k^\star)\right) \cdot P\!\left(Y = y'^\star \,\big|\, u, \mathrm{do}(X_k = x_k')\right) \, dP_U(u),$$

which is Pearl's individual-level PNS in an SCM with shared exogenous noise.

The cell below verifies this against the `pns_*` values computed earlier in this notebook for all four (person, target) cases. **Heads up:** three of four match exactly; for Bob/credit there is a small discrepancy explained below.

In [12]:
def compute_pci_single_suspect(factual_gender, factual_credit, target):
    """
    PCI with S = {target}, W = ∅ — the equivalence-claim configuration.

    Single (C, T) configuration → no Γ averaging.  Reduces to:
       ∫ P(Y=0 | u, do(target=factual), other=factual)
         × P(Y=1 | u, do(target=alt),     other=factual) dP_U(u),
    integrated over u_check regions.
    """
    if target == 'gender':
        suff_gender, nec_gender = factual_gender, 1 - factual_gender
        suff_credit, nec_credit = factual_credit, factual_credit
    elif target == 'credit':
        suff_gender, nec_gender = factual_gender, factual_gender
        suff_credit, nec_credit = factual_credit, 1 - factual_credit
    else:
        raise ValueError(f"unknown target: {target}")

    total = 0.0
    for region_name, (region_prob, check_by_gender) in u_regions.items():
        # Sufficiency world
        suff_check = check_by_gender[suff_gender]
        if suff_check == 0:
            P_suf_y0 = 1.0
        else:
            cf = suff_check * (1 - suff_credit)
            P_suf_y0 = 1.0 - loan_prob[(suff_gender, cf)]

        # Necessity world
        nec_check = check_by_gender[nec_gender]
        if nec_check == 0:
            P_nec_y1 = 0.0
        else:
            cf = nec_check * (1 - nec_credit)
            P_nec_y1 = loan_prob[(nec_gender, cf)]

        total += region_prob * P_suf_y0 * P_nec_y1

    return total


# ── Compute single-suspect PCI for all four cases ────────────────────────────
cases = [
    ("Alice", 0, 0, 'gender', pns_alice_gender),
    ("Alice", 0, 0, 'credit', pns_alice_credit),
    ("Bob",   1, 0, 'gender', pns_bob_gender),
    ("Bob",   1, 0, 'credit', pns_bob_credit),
]

print(f"{'Person':<6} {'Target':<7} {'PCI(S={X_k}, W=∅)':<20} {'pns_* (notebook)':<18} {'match?'}")
print("-" * 72)
for person, fg, fc, target, pns_value in cases:
    pci_val = compute_pci_single_suspect(fg, fc, target)
    diff = pci_val - pns_value
    if abs(diff) < 1e-10:
        status = "✓ exact"
    else:
        status = f"differ by {diff:+.6f}"
    print(f"{person:<6} {target:<7} {pci_val:<20.6f} {pns_value:<18.6f} {status}")

Person Target  PCI(S={X_k}, W=∅)    pns_* (notebook)   match?
------------------------------------------------------------------------
Alice  gender  0.045000             0.045000           ✓ exact
Alice  credit  0.180000             0.180000           ✓ exact
Bob    gender  0.000000             0.000000           ✓ exact
Bob    credit  0.855000             0.859500           differ by -0.004500


### Why Bob/credit shows a small discrepancy

The `pns_bob_credit` value computed earlier in this notebook factorises:

$$\mathrm{pns\_bob\_credit} = \big(1 - p_\text{loan}(\text{male, bad})\big) \cdot p_\text{loan}(\text{male, good}) = \mathbb{E}_u[P^s] \cdot \mathbb{E}_u[P^n].$$

PCI integrates the **product**:

$$\mathrm{PCI}_\text{credit}(\text{Bob}) = \int P^s(u) \cdot P^n(u) \, dP_U(u) = \mathbb{E}_u[P^s \cdot P^n].$$

These differ by $\mathrm{Cov}_u(P^s, P^n)$.  For Bob/credit both factors vary in $u_\text{check}$:

- $P^s(u) = P(\text{loan}=0 \mid u_\text{check}, \text{male, bad})$: $0.95$ when checked, $1$ when not.
- $P^n(u) = P(\text{loan}=1 \mid u_\text{check}, \text{male, good})$: $1.0$ when checked, $0$ when not.

The factors are negatively correlated through $u_\text{check}$ (when Bob is not checked, $P^s$ peaks but $P^n = 0$), so the factorised version overshoots by $\approx 0.0045$.

**Pearl's actual individual-level PNS**, defined as $P(Y_{c'}=1, Y_{c^\star}=0)$ in an SCM with shared exogenous noise, is the *integrated* form — exactly what PCI computes.  So the discrepancy is the factorised formula, not PCI: PCI delivers $\int P^s \cdot P^n \, dP_U$ correctly, and the equivalence claim holds.

The other three cases match exactly because at least one factor is constant in $u$:
- Alice/gender, Alice/credit: $P^s = 1$ identically (female + bad denied in every $u$).
- Bob/gender: $P^n = 0$ identically (female + bad approved with probability $0$).

Whenever one factor is constant, $\mathbb{E}[P^s \cdot P^n] = \mathbb{E}[P^s] \cdot \mathbb{E}[P^n]$ trivially.

## SHAP values for Alice and Bob

Plain SHAP explains the model prediction $f(g, c) = P(\text{loan}=1 \mid g, c)$.
With $|N|=2$ features the Shapley formula~\eqref{eq:shapley} reduces to:

$$\phi_i = \tfrac{1}{2}\bigl[v(\{i\}) - v(\emptyset)\bigr]
          + \tfrac{1}{2}\bigl[v(N) - v(N \setminus \{i\})\bigr]$$

Missing features are marginalized over the **marginal** distribution
$P(\text{gender}) = P(\text{credit}) = \tfrac{1}{2}$ (uniform prior).

We compute SHAP for the **rejection** outcome, i.e.\ for
$g(x) = P(\text{loan}=0 \mid x) = 1 - f(x)$.  Because $g = 1-f$, the
characteristic functions satisfy $v_g(S) = 1 - v_f(S)$, so

$$\phi^g_i = -\phi^f_i$$

We therefore compute $\phi^f$ and flip signs.  Positive $\phi^g_i$ means
feature $i$ pushes the rejection probability up; negative means it pushes it down.

In [13]:
def shap_rejection(factual_gender, factual_credit):
    """
    Exact SHAP values for g(x) = P(loan=0 | x), the rejection probability.

    With |N|=2 and uniform P(gender)=P(credit)=0.5 the formula is:
      phi_i = 0.5*(v({i}) - v({})) + 0.5*(v(N) - v(N\{i}))
    where v(S) = E_{X_bar_S}[f(x_S, X_bar_S)] for f = P(loan=1|.).
    Flip sign at the end for g = 1-f.
    """
    g, c = factual_gender, factual_credit

    # ── characteristic function for f = P(loan=1) ────────────────────────────
    v_empty   = 0.25 * sum(p_loan(gi, ci) for gi in [0,1] for ci in [0,1])
    v_gender  = 0.5 * p_loan(g, 0) + 0.5 * p_loan(g, 1)   # marginalise credit
    v_credit  = 0.5 * p_loan(0, c) + 0.5 * p_loan(1, c)   # marginalise gender
    v_both    = p_loan(g, c)

    phi_f_gender = 0.5*(v_gender - v_empty) + 0.5*(v_both - v_credit)
    phi_f_credit = 0.5*(v_credit - v_empty) + 0.5*(v_both - v_gender)

    # Flip for g = 1-f
    phi_g_gender = -phi_f_gender
    phi_g_credit = -phi_f_credit

    return phi_g_gender, phi_g_credit, v_empty, v_gender, v_credit, v_both


# ── Alice (gender=F=0, credit=bad=0) ─────────────────────────────────────────
a_phi_g, a_phi_c, v0, vg_a, vc, vb_a = shap_rejection(0, 0)
print("Alice (gender=F, credit=bad)")
print(f"  v({{}})         = {v0:.5f}  (baseline: E[P(loan=1)])")
print(f"  v({{gender=F}})  = {vg_a:.5f}")
print(f"  v({{credit=bad}})= {vc:.5f}")
print(f"  v(N)           = {vb_a:.5f}")
print(f"  phi_g (approval) = {-a_phi_g:.5f}  ->  phi_g (rejection) = {a_phi_g:.5f}")
print(f"  phi_c (approval) = {-a_phi_c:.5f}  ->  phi_c (rejection) = {a_phi_c:.5f}")
print(f"  check efficiency: {a_phi_g + a_phi_c:.5f} == {(1-vb_a) - (1-v0):.5f}")

print()

# ── Bob (gender=M=1, credit=bad=0) ───────────────────────────────────────────
b_phi_g, b_phi_c, _, vg_b, _, vb_b = shap_rejection(1, 0)
print("Bob (gender=M, credit=bad)")
print(f"  v({{}})          = {v0:.5f}  (same baseline)")
print(f"  v({{gender=M}})   = {vg_b:.5f}")
print(f"  v({{credit=bad}}) = {vc:.5f}")
print(f"  v(N)            = {vb_b:.5f}")
print(f"  phi_g (approval) = {-b_phi_g:.5f}  ->  phi_g (rejection) = {b_phi_g:.5f}")
print(f"  phi_c (approval) = {-b_phi_c:.5f}  ->  phi_c (rejection) = {b_phi_c:.5f}")
print(f"  check efficiency: {b_phi_g + b_phi_c:.5f} == {(1-vb_b) - (1-v0):.5f}")

Alice (gender=F, credit=bad)
  v({})         = 0.28125  (baseline: E[P(loan=1)])
  v({gender=F})  = 0.09000
  v({credit=bad})= 0.02250
  v(N)           = 0.00000
  phi_g (approval) = -0.10687  ->  phi_g (rejection) = 0.10687
  phi_c (approval) = -0.17438  ->  phi_c (rejection) = 0.17438
  check efficiency: 0.28125 == 0.28125

Bob (gender=M, credit=bad)
  v({})          = 0.28125  (same baseline)
  v({gender=M})   = 0.47250
  v({credit=bad}) = 0.02250
  v(N)            = 0.04500
  phi_g (approval) = 0.10688  ->  phi_g (rejection) = -0.10688
  phi_c (approval) = -0.34313  ->  phi_c (rejection) = 0.34313
  check efficiency: 0.23625 == 0.23625


<>:2: SyntaxWarning: invalid escape sequence '\{'
<>:2: SyntaxWarning: invalid escape sequence '\{'
/tmp/ipykernel_75053/1635022816.py:2: SyntaxWarning: invalid escape sequence '\{'
  """


In [14]:
# ── Four-method comparison table ─────────────────────────────────────────────
rows = [
    ("Alice", "gender", f"{pns_alice_gender:.3f}", f"{a_phi_g:.3f}",
     f"{alice_g_nw:.3f}", f"{alice_g_w:.3f}"),
    ("Alice", "credit", f"{pns_alice_credit:.3f}", f"{a_phi_c:.3f}",
     f"{alice_c_nw:.3f}", f"{alice_c_w:.3f}"),
    ("Bob",   "gender", f"{pns_bob_gender:.3f}",   f"{b_phi_g:.3f}",
     f"{bob_g_nw:.3f}",   f"{bob_g_w:.3f}"),
    ("Bob",   "credit", f"{pns_bob_credit:.3f}",   f"{b_phi_c:.3f}",
     f"{bob_c_nw:.3f}",   f"{bob_c_w:.3f}"),
]

df4 = pd.DataFrame(rows, columns=[
    "Person", "Feature", "PNS", "SHAP", "PCI (no wit.)", "PCI (wit.)"])
print(df4.to_string(index=False))

# ── Desiderata check (desiderata use |R(·)| magnitudes; D-B1 is strict > 0) ──
print()
checks = [
    ("D-A1   |R(gender)|>0 Alice",
     pns_alice_gender > 0, abs(a_phi_g) > 0, alice_g_nw > 0, alice_g_w > 0),
    ("D-A2   |R(credit)|>0 Alice",
     pns_alice_credit > 0, abs(a_phi_c) > 0, alice_c_nw > 0, alice_c_w > 0),
    ("D-A-rank  |gender|>|credit| Alice",
     pns_alice_gender > pns_alice_credit,
     abs(a_phi_g) > abs(a_phi_c),
     alice_g_nw > alice_c_nw,
     alice_g_w  > alice_c_w),
    ("D-B1   |R(gender)|>0 Bob  (strict)",
     pns_bob_gender > 0, abs(b_phi_g) > 0, bob_g_nw > 0, bob_g_w > 0),
    ("D-B2   |R(credit)|>0 Bob",
     pns_bob_credit > 0, abs(b_phi_c) > 0, bob_c_nw > 0, bob_c_w > 0),
    ("D-B-rank  |credit|>|gender| Bob",
     pns_bob_credit > pns_bob_gender,
     abs(b_phi_c) > abs(b_phi_g),
     bob_c_nw > bob_g_nw,
     bob_c_w  > bob_g_w),
    ("D-comp  |gender Alice|>|gender Bob|",
     pns_alice_gender > pns_bob_gender,
     abs(a_phi_g) > abs(b_phi_g),
     alice_g_nw > bob_g_nw,
     alice_g_w  > bob_g_w),
]

header = f"{'Desideratum':<38} {'PNS':>4} {'SHAP':>5} {'PCI-nw':>7} {'PCI-w':>6}"
print(header)
print("-" * len(header))
for label, pns_ok, shap_ok, pci_nw_ok, pci_w_ok in checks:
    tick = lambda b: "✓" if b else "✗"
    print(f"{label:<38} {tick(pns_ok):>4} {tick(shap_ok):>5} {tick(pci_nw_ok):>7} {tick(pci_w_ok):>6}")

Person Feature   PNS   SHAP PCI (no wit.) PCI (wit.)
 Alice  gender 0.045  0.107         0.210      0.263
 Alice  credit 0.180  0.174         0.240      0.199
   Bob  gender 0.000 -0.107         0.038      0.019
   Bob  credit 0.859  0.343         0.228      0.119

Desideratum                             PNS  SHAP  PCI-nw  PCI-w
----------------------------------------------------------------
D-A1   |R(gender)|>0 Alice                ✓     ✓       ✓      ✓
D-A2   |R(credit)|>0 Alice                ✓     ✓       ✓      ✓
D-A-rank  |gender|>|credit| Alice         ✗     ✗       ✗      ✓
D-B1   |R(gender)|>0 Bob  (strict)        ✗     ✓       ✓      ✓
D-B2   |R(credit)|>0 Bob                  ✓     ✓       ✓      ✓
D-B-rank  |credit|>|gender| Bob           ✓     ✓       ✓      ✓
D-comp  |gender Alice|>|gender Bob|       ✓     ✗       ✓      ✓


## Causal SHAP values for Alice and Bob (Heskes et al. 2020)

### Definition

Causal SHAP replaces the characteristic function with an interventional expectation:

$$v_\text{causal}(S) = \mathbb{E}[f(X) \mid do(X_S = x_S)]
= \int P(X_{\bar{S}} \mid do(X_S = x_S))\, f(x_S, X_{\bar{S}})\, dX_{\bar{S}}$$

Plain SHAP uses the observational marginal $P(X_{\bar{S}})$ for dropped features.
Causal SHAP uses the **interventional** distribution
$P(X_{\bar{S}} \mid do(X_S = x_S))$.  The two agree whenever the dropped features
are not causally downstream of the coalition.

### What `do()` changes and what it does not

$do(X_S = x_S)$ surgically removes all arrows *into* $X_S$.
As a consequence, **only descendants of $S$ in the causal graph over the model
inputs have their distribution altered**:

$$P(X_j \mid do(X_S = x_S)) = P(X_j)
\quad \text{if } X_j \notin S \text{ is not a descendant of any } X_i \in S$$

This is exactly where Causal SHAP differs from plain SHAP: when a dropped
model input is causally downstream of a coalition member, plain SHAP marginalizes
it with the wrong (non-interventional) distribution.

### The critical point: what are the model inputs?

The full OBCB SCM has downstream variables: `check`, `check_failed`,
`loan_if_checked`, `loan`.  In the SCM causal graph, `check_failed` **is** a
descendant of both `gender` (via `check`) and `credit`.  So if `check_failed`
were a model input, Causal SHAP and plain SHAP would give different results.

However, the model we apply SHAP to is:

$$f(\text{gender}, \text{credit}) = P(\text{loan}=1 \mid \text{gender}, \text{credit})$$

The feature set is $N = \{\text{gender},\, \text{credit}\}$ **only**.
The intermediate variables are integrated out *inside* $f$ — they are not
inputs being marginalized by SHAP.  The Causal SHAP formula marginalizes only
over dropped **model inputs** $X_{\bar{S}}$.  With $|N|=2$, the only dropped
features are `gender` or `credit` (whichever is not in the coalition).

### Why the two are identical for this feature set

In the causal graph **restricted to the model inputs** $\{\text{gender}, \text{credit}\}$,
both are independent exogenous roots — neither is a descendant of the other:

```
gender  ~  Bernoulli(0.5)   (root — no parents, no causal link to credit)
credit  ~  Bernoulli(0.5)   (root — no parents, no causal link to gender)
```

Since neither dropped feature is a descendant of the coalition:

$$P(\text{credit} \mid do(\text{gender}=g)) = P(\text{credit})$$
$$P(\text{gender} \mid do(\text{credit}=c)) = P(\text{gender})$$

Therefore $v_\text{causal}(S) = v_\text{plain}(S)$ for every coalition, and

$$\phi^\text{causal}_i = \phi^\text{plain}_i \quad \text{exactly}$$

### Why this matters

The equality holds for this feature set ($N = \{\text{gender},\text{credit}\}$),
not in general.  If the practitioner included `check_failed` as a model input,
Causal SHAP would differ from plain SHAP because `check_failed` is downstream of
both features.

More importantly: the D-A-rank and D-comp failures persist under Causal SHAP
because the root cause is not feature correlation or causal graph misspecification
— it is that both methods explain $f(\text{gender}, \text{credit})$, a predicted
output averaged over the model's internal noise, rather than the individual
factual outcome of this specific person.

In [15]:
def causal_shap_rejection(factual_gender, factual_credit):
    """
    Causal SHAP (Heskes et al. 2020) for g(x) = P(loan=0 | x).

    v_causal(S) = E[f(X) | do(X_S = x_S)]

    For OBCB: gender and credit are independent exogenous variables.
    do(gender=g) does not change P(credit), and vice versa.
    Therefore v_causal(S) = v_plain(S) for all S.
    """
    g, c = factual_gender, factual_credit

    # ── Causal characteristic function ────────────────────────────────────────
    # v_causal({}) = E[f(X)] — baseline, same as plain SHAP
    v_empty = 0.25 * sum(p_loan(gi, ci) for gi in [0,1] for ci in [0,1])

    # v_causal({gender=g}): do(gender=g), credit unaffected (independent)
    #   P(credit | do(gender=g)) = P(credit)  →  same as plain SHAP
    v_gender_causal = 0.5 * p_loan(g, 0) + 0.5 * p_loan(g, 1)

    # v_causal({credit=c}): do(credit=c), gender unaffected (independent)
    #   P(gender | do(credit=c)) = P(gender)  →  same as plain SHAP
    v_credit_causal = 0.5 * p_loan(0, c) + 0.5 * p_loan(1, c)

    # v_causal({gender=g, credit=c}) = f(g, c)  — same as plain SHAP
    v_both = p_loan(g, c)

    phi_f_gender = 0.5*(v_gender_causal - v_empty) + 0.5*(v_both - v_credit_causal)
    phi_f_credit = 0.5*(v_credit_causal - v_empty) + 0.5*(v_both - v_gender_causal)

    phi_g_gender = -phi_f_gender   # flip for rejection
    phi_g_credit = -phi_f_credit

    return phi_g_gender, phi_g_credit, v_empty, v_gender_causal, v_credit_causal, v_both


# ── Compute and compare ───────────────────────────────────────────────────────
a_cg, a_cc, *_ = causal_shap_rejection(0, 0)
b_cg, b_cc, *_ = causal_shap_rejection(1, 0)

print("Causal SHAP vs plain SHAP for rejection P(loan=0|x):")
print(f"  Alice gender:  plain={a_phi_g:.5f}  causal={a_cg:.5f}  equal={abs(a_phi_g-a_cg)<1e-10}")
print(f"  Alice credit:  plain={a_phi_c:.5f}  causal={a_cc:.5f}  equal={abs(a_phi_c-a_cc)<1e-10}")
print(f"  Bob   gender:  plain={b_phi_g:.5f}  causal={b_cg:.5f}  equal={abs(b_phi_g-b_cg)<1e-10}")
print(f"  Bob   credit:  plain={b_phi_c:.5f}  causal={b_cc:.5f}  equal={abs(b_phi_c-b_cc)<1e-10}")

print()
print("Desiderata (|·| magnitudes, D-B1 strict > 0):")
checks_causal = [
    ("D-A1   |R(gender)|>0 Alice",       abs(a_cg) > 0),
    ("D-A2   |R(credit)|>0 Alice",       abs(a_cc) > 0),
    ("D-A-rank |gender|>|credit| Alice", abs(a_cg) > abs(a_cc)),
    ("D-B1   |R(gender)|>0 Bob",         abs(b_cg) > 0),
    ("D-B2   |R(credit)|>0 Bob",         abs(b_cc) > 0),
    ("D-B-rank |credit|>|gender| Bob",   abs(b_cc) > abs(b_cg)),
    ("D-comp |gender Alice|>|gender Bob|", abs(a_cg) > abs(b_cg)),
]
for label, ok in checks_causal:
    print(f"  {label:<38} {'✓' if ok else '✗'}")

Causal SHAP vs plain SHAP for rejection P(loan=0|x):
  Alice gender:  plain=0.10687  causal=0.10687  equal=True
  Alice credit:  plain=0.17438  causal=0.17438  equal=True
  Bob   gender:  plain=-0.10688  causal=-0.10688  equal=True
  Bob   credit:  plain=0.34313  causal=0.34313  equal=True

Desiderata (|·| magnitudes, D-B1 strict > 0):
  D-A1   |R(gender)|>0 Alice             ✓
  D-A2   |R(credit)|>0 Alice             ✓
  D-A-rank |gender|>|credit| Alice       ✗
  D-B1   |R(gender)|>0 Bob               ✓
  D-B2   |R(credit)|>0 Bob               ✓
  D-B-rank |credit|>|gender| Bob         ✓
  D-comp |gender Alice|>|gender Bob|     ✗


## Extended feature set: Causal SHAP including the mediator

The argument that Causal SHAP coincides with plain SHAP on this example
relies on the choice of feature set $N=\{\text{gender},\text{credit}\}$.
PCI, by contrast, does access the mediator `check_failed` — through the
witness mechanism. To make the comparison even, we now apply both plain
SHAP and Causal SHAP to the **3-feature** game

$$N \;=\; \{\text{gender},\; \text{credit},\; \text{check\_failed}\},$$

i.e.\ we treat `check_failed` as an observable model input, the same status
that the mediator $M$ has in the signal-with-mediation example.

### The model $\tilde f$

We use the natural observational extension

$$\tilde f(g,c,cf) \;=\; \mathbb{E}[\text{loan} \mid \text{gender}=g,\,\text{credit}=c,\,\text{check\_failed}=cf].$$

Under the OBCB SCM, `check_failed = check · (1 - credit)`, so the cell
$(c{=}1,cf{=}1)$ has zero observational mass and $\tilde f$ is **undefined**
there. We refuse to extrapolate and let the value be `NaN`.

| $g$ | $c$ | $cf$ | $\tilde f(g,c,cf)$ |
|-----|-----|------|--------------------|
| any | 0   | 0    | $0$ ($cf=0\wedge c=0\Rightarrow check=0\Rightarrow loan=0$) |
| $g$ | 0   | 1    | $\text{loan\_prob}[g,1]$ ($cf=1\wedge c=0\Rightarrow check=1$) |
| $g$ | 1   | 0    | $p_\text{check}[g]\cdot\text{loan\_prob}[g,0]$ |
| any | 1   | 1    | **undefined** (impossible under the SCM) |

### Heskes' definition applied here

Definition~\ref{def:causalshap} (Heskes et al.\ 2020):

$$v_\mathrm{causal}(S) \;=\; \mathbb{E}\!\left[\tilde f(X)\,\big|\,do(X_S = x_S^\star)\right],$$

where dropped features are split into non-descendants of $S$ (sampled
from their marginals) and descendants of $S$ (sampled from their
$do$-conditional). In the causal graph over the model inputs:

- gender, credit are exogenous roots,
- check\_failed is a descendant of both, with $cf = check\cdot(1-c)$,
  $check\sim\mathrm{Bern}(p_\text{check}[g])$.

So under $do(X_S=x_S^\star)$:

$$P(cf=1\mid do(X_S=x_S^\star),g,c)
   \;=\; p_\text{check}[g]\cdot(1-c),$$

with $g$ and $c$ either fixed (if in $S$) or sampled from their
marginals. When $cf$ itself is in $S$, $cf$ is fixed at $cf^\star$ and
the structural relation is severed — at which point a coalition can
require $\tilde f(g,c{=}1,cf{=}1)$, hitting the undefined cell.

### What we expect

For Alice ($g{=}0,c{=}0,cf{=}0$), the impossible cell never receives
positive weight: every coalition that drops $cf$ samples it from the
SCM, where $(c{=}1,cf{=}1)$ has mass zero; coalitions with $cf\in S$
fix $cf^\star=0$, so they only ever query $(c,0)$. Causal SHAP is
well-defined.

For Bob ($g{=}1,c{=}0,cf{=}1$), coalitions that fix $cf^\star=1$ and
let $c$ be sampled marginally — i.e.\ $S=\{cf\}$ and $S=\{g,cf\}$ — query
$\tilde f(\cdot,1,1)$ with positive weight. Bob's Causal SHAP is therefore
**undefined** without an extrapolation rule.


In [16]:
# ── Extended 3-feature observational model ─────────────────────────────────
#
# Strict observational extension. The cell (c=1, cf=1) has zero observational
# mass under the OBCB SCM and is therefore undefined; we return NaN there.

import math

def f_obs(g, c, cf):
    if c == 0 and cf == 0:
        return 0.0
    if c == 0 and cf == 1:
        return loan_prob[(g, 1)]
    if c == 1 and cf == 0:
        return p_check[g] * loan_prob[(g, 0)]
    return math.nan  # (c=1, cf=1) is impossible under the SCM


def p_cf_marg(cf):
    p1 = sum(0.5 * 0.5 * p_check[g] * (1 - c) for g in [0, 1] for c in [0, 1])
    return p1 if cf == 1 else (1 - p1)


# ── Causal SHAP value function (Heskes et al. 2020) ──────────────────────
#
# v_causal(S) = E[f(X) | do(X_S = x_S^*)].  Implementation skips zero-weight
# terms so that NaN cells are only ever poisoned by *positive*-weight queries
# (i.e., when the do-distribution genuinely puts mass on the undefined cell).

def v_causal(S, x_star, fhat):
    g_iter = [x_star["g"]] if "g" in S else [0, 1]
    c_iter = [x_star["c"]] if "c" in S else [0, 1]
    p_g = (lambda g: 1.0) if "g" in S else (lambda g: 0.5)
    p_c = (lambda c: 1.0) if "c" in S else (lambda c: 0.5)

    total = 0.0
    weight_check = 0.0
    for g in g_iter:
        for c in c_iter:
            if "cf" in S:
                cf_iter = [x_star["cf"]]
                p_cf_local = lambda cf: 1.0
            else:
                cf_iter = [0, 1]
                p1 = p_check[g] * (1 - c)
                p_cf_local = (lambda p1=p1: (lambda cf: p1 if cf == 1 else (1 - p1)))()
            for cf in cf_iter:
                w = p_g(g) * p_c(c) * p_cf_local(cf)
                if w == 0:
                    continue  # skip zero-mass terms (avoid 0 * NaN poisoning)
                total += w * fhat(g, c, cf)
                weight_check += w
    assert abs(weight_check - 1.0) < 1e-9, weight_check
    return total


def v_plain(S, x_star, fhat):
    g_iter  = [x_star["g"]]  if "g"  in S else [0, 1]
    c_iter  = [x_star["c"]]  if "c"  in S else [0, 1]
    cf_iter = [x_star["cf"]] if "cf" in S else [0, 1]
    p_g  = (lambda g: 1.0)  if "g"  in S else (lambda g: 0.5)
    p_c  = (lambda c: 1.0)  if "c"  in S else (lambda c: 0.5)
    p_cf = (lambda cf: 1.0) if "cf" in S else p_cf_marg

    total = 0.0
    for g in g_iter:
        for c in c_iter:
            for cf in cf_iter:
                w = p_g(g) * p_c(c) * p_cf(cf)
                if w == 0:
                    continue
                total += w * fhat(g, c, cf)
    return total


print("f_obs cells:")
for g in [0, 1]:
    for c in [0, 1]:
        for cf in [0, 1]:
            print(f"  f({g},{c},{cf}) = {f_obs(g, c, cf)}")
print()
print(f"P(cf=0) = {p_cf_marg(0):.4f},  P(cf=1) = {p_cf_marg(1):.4f}")


f_obs cells:
  f(0,0,0) = 0.0
  f(0,0,1) = 0.0
  f(0,1,0) = 0.18000000000000002
  f(0,1,1) = nan
  f(1,0,0) = 0.0
  f(1,0,1) = 0.05
  f(1,1,0) = 0.9
  f(1,1,1) = nan

P(cf=0) = 0.7250,  P(cf=1) = 0.2750


In [17]:
# ── Shapley computation over the 3-feature game ──────────────────────────────

from itertools import combinations
from math import factorial, isnan

FEATURES = ("g", "c", "cf")

def powerset(iterable):
    s = list(iterable)
    return [frozenset(c) for r in range(len(s) + 1)
            for c in combinations(s, r)]

def shapley_with_trace(v_fn, x_star, fhat, target):
    n = len(FEATURES)
    others = [f for f in FEATURES if f != target]
    phi = 0.0
    trace = []
    for S in powerset(others):
        s = len(S)
        weight = factorial(s) * factorial(n - s - 1) / factorial(n)
        S_set = set(S)
        v_with    = v_fn(S_set | {target}, x_star, fhat)
        v_without = v_fn(S_set,             x_star, fhat)
        phi += weight * (v_with - v_without)
        trace.append((tuple(sorted(S_set)), v_without, tuple(sorted(S_set | {target})), v_with))
    return phi, trace


def shap_table(x_star, label):
    print(f"\n=== {label}  factual = {x_star} ===")
    out = {}
    nan_coals_causal = set()
    for target in FEATURES:
        phi_p, _ = shapley_with_trace(v_plain,  x_star, f_obs, target)
        phi_c, trace_c = shapley_with_trace(v_causal, x_star, f_obs, target)
        out[("plain",  target)] = phi_p
        out[("causal", target)] = phi_c
        for (S_a, v_a, S_b, v_b) in trace_c:
            if isinstance(v_a, float) and isnan(v_a): nan_coals_causal.add(S_a)
            if isinstance(v_b, float) and isnan(v_b): nan_coals_causal.add(S_b)
    if nan_coals_causal:
        print(f"  NaN-producing coalitions under v_causal: {sorted(nan_coals_causal)}")
    full = f_obs(x_star['g'], x_star['c'], x_star['cf'])
    base_p = v_plain(set(),  x_star, f_obs)
    base_c = v_causal(set(), x_star, f_obs)
    print(f"  full f-hat(x*)      = {full}")
    print(f"  v_plain({{}})         = {base_p}")
    print(f"  v_causal({{}})        = {base_c}")
    print()
    print(f"  Feature       phi_plain      phi_causal")
    for f in FEATURES:
        pp = out[("plain", f)]
        pc = out[("causal", f)]
        pp_s = "  NaN  " if (isinstance(pp, float) and isnan(pp)) else f"{pp:+.5f}"
        pc_s = "  NaN  " if (isinstance(pc, float) and isnan(pc)) else f"{pc:+.5f}"
        print(f"  {f:<12}  {pp_s}      {pc_s}")
    return out


alice_x = {"g": 0, "c": 0, "cf": 0}    # female, bad, not checked
bob_x   = {"g": 1, "c": 0, "cf": 1}    # male, bad, checked, check_failed = 1

alice_out = shap_table(alice_x, "Alice")
bob_out   = shap_table(bob_x,   "Bob")



=== Alice  factual = {'g': 0, 'c': 0, 'cf': 0} ===
  full f-hat(x*)      = 0.0
  v_plain({})         = nan
  v_causal({})        = 0.28125

  Feature       phi_plain      phi_causal
  g               NaN        -0.09750
  c               NaN        -0.17625
  cf              NaN        -0.00750

=== Bob  factual = {'g': 1, 'c': 0, 'cf': 1} ===
  NaN-producing coalitions under v_causal: [('cf',), ('cf', 'g')]
  full f-hat(x*)      = 0.05
  v_plain({})         = nan
  v_causal({})        = 0.28125

  Feature       phi_plain      phi_causal
  g               NaN          NaN  
  c               NaN          NaN  
  cf              NaN          NaN  


In [18]:
# ── Desiderata under the 3-feature game (f_obs strict) ───────────────────

from math import isnan

def cmp(a, b):
    if (isinstance(a, float) and isnan(a)) or (isinstance(b, float) and isnan(b)):
        return "undef"
    return "PASS" if a > b else "FAIL"

def gt0(a):
    if isinstance(a, float) and isnan(a):
        return "undef"
    return "PASS" if a > 1e-9 else "FAIL"

def mag(out, m, f):
    v = out[(m, f)]
    return float("nan") if (isinstance(v, float) and isnan(v)) else abs(v)

def report(method, alice, bob):
    A = {f: mag(alice, method, f) for f in FEATURES}
    B = {f: mag(bob,   method, f) for f in FEATURES}
    print(f"\n--- Desiderata under {method} ---")
    rows = [
        ("D-A1   |gender|>0 Alice",              gt0(A["g"])),
        ("D-A2   |credit|>0 Alice",              gt0(A["c"])),
        ("D-A-rank gender>credit Alice",         cmp(A["g"], A["c"])),
        ("D-B1   |gender|>0 Bob",                gt0(B["g"])),
        ("D-B2   |credit|>0 Bob",                gt0(B["c"])),
        ("D-B-rank credit>gender Bob",           cmp(B["c"], B["g"])),
        ("D-comp |gender|_Alice > |gender|_Bob", cmp(A["g"], B["g"])),
    ]
    for label, status in rows:
        print(f"  {label:<42} {status}")

report("plain",  alice_out, bob_out)
report("causal", alice_out, bob_out)


# ── Magnitudes summary table ─────────────────────────────────────────────
import pandas as pd
def fmtm(out, m, f):
    v = out[(m, f)]
    return "NaN" if (isinstance(v, float) and isnan(v)) else f"{abs(v):.3f}"

rows = []
for method in ("plain", "causal"):
    rows.append((
        method,
        fmtm(alice_out, method, "g"),
        fmtm(alice_out, method, "c"),
        fmtm(alice_out, method, "cf"),
        fmtm(bob_out,   method, "g"),
        fmtm(bob_out,   method, "c"),
        fmtm(bob_out,   method, "cf"),
    ))
df = pd.DataFrame(rows, columns=["method", "A:|g|", "A:|c|", "A:|cf|", "B:|g|", "B:|c|", "B:|cf|"])
print()
print(df.to_string(index=False))



--- Desiderata under plain ---
  D-A1   |gender|>0 Alice                    undef
  D-A2   |credit|>0 Alice                    undef
  D-A-rank gender>credit Alice               undef
  D-B1   |gender|>0 Bob                      undef
  D-B2   |credit|>0 Bob                      undef
  D-B-rank credit>gender Bob                 undef
  D-comp |gender|_Alice > |gender|_Bob       undef

--- Desiderata under causal ---
  D-A1   |gender|>0 Alice                    PASS
  D-A2   |credit|>0 Alice                    PASS
  D-A-rank gender>credit Alice               FAIL
  D-B1   |gender|>0 Bob                      undef
  D-B2   |credit|>0 Bob                      undef
  D-B-rank credit>gender Bob                 undef
  D-comp |gender|_Alice > |gender|_Bob       undef

method A:|g| A:|c| A:|cf| B:|g| B:|c| B:|cf|
 plain   NaN   NaN    NaN   NaN   NaN    NaN
causal 0.097 0.176  0.007   NaN   NaN    NaN
